In [0]:
import pyspark.sql.functions as F
import requests
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
csv_path = "/Volumes/dbr_dev/w_k_palczewski/raw_data/Data.csv"

df_raw = spark.read.csv(csv_path, header=True, inferSchema=True)
df_raw.show()

In [0]:
# Data is flawed duplicate rows or Players with missing teams
cond_brackets = df_raw["Club"].rlike(r"^\([A-Za-z]{3}\)$")
df_cleaned = df_raw.filter(cond_brackets)

df_cleaned.show()
df_cleaned.write.format("delta").option("delta.columnMapping.mode", "name").mode("overwrite").saveAsTable("dbr_dev.w_k_palczewski.football_scorers")

In [0]:
df = spark.read.table("dbr_dev.w_k_palczewski.football_scorers")
filtered_df = df.select("Player Names", "Club", "League", "Goals") \
                .filter(df["Goals"] > 20)
display(filtered_df)

Databricks visualization. Run in Databricks to view.

In [0]:
grouped_df = df.groupBy("Club").agg(F.sum("Goals").alias("Total_Goals")).orderBy(F.desc("Total_Goals"))
display(grouped_df)

Databricks visualization. Run in Databricks to view.

In [0]:
df_clubs = df.select("Player Names", "Club").distinct()
df_goals = df.select("Player Names", "Goals").distinct()

joined_df = df_clubs.join(df_goals, "Player Names", "inner")
display(joined_df)

Databricks visualization. Run in Databricks to view.

In [0]:
url = "https://raw.githubusercontent.com/openfootball/football.json/master/2015-16/en.1.json"
response = requests.get(url)
api_data = response.json()

matches = api_data.get("matches", [])

processed_matches = []

for m in matches:
    round_name = m.get("round", "")
    date = m.get("date", "")
    team_1 = m.get("team1", "")
    team_2 = m.get("team2", "")

    score_ft = m.get("score", {}).get("ft", [None, None])
    score_1 = score_ft[0] if len(score_ft) > 0 else None
    score_2 = score_ft[1] if len(score_ft) > 1 else None
    
    processed_matches.append((round_name, date, team_1, team_2, score_1, score_2))

schema = StructType([
    StructField("Round", StringType(), True),
    StructField("Date", StringType(), True),
    StructField("Team_1", StringType(), True),
    StructField("Team_2", StringType(), True),
    StructField("Score_1", IntegerType(), True),
    StructField("Score_2", IntegerType(), True)
])

spark_api_df = spark.createDataFrame(processed_matches, schema=schema)

spark_api_df.write.format("delta").mode("overwrite").saveAsTable("dbr_dev.w_k_palczewski.api_premier_league")
display(spark_api_df)